# Install Required Libraries

In [ ]:
# Install necessary libraries
!pip install -q transformers datasets evaluate
!pip install -q pyttsx3  # For voice synthesis (text-to-speech)
!pip install -q gradio  # For creating an upload interface


In [ ]:
!pip install gTTS # Install the gTTS library

In [ ]:
!sudo apt install espeak # Install espeak using apt package manager

In [ ]:
!gdown "1_gqdScnGPxDujbsHKiqWHyRaFhKnzCvU"
# Downloading trained DeepFake model

In [ ]:
# prompt: !unzip deepfake_model.zip into '/content/deepfake_model'

!unzip deepfake_model.zip -d /content/deepfake_model


# Import Libraries and Load the Model

In [ ]:
# Import Libraries and Load the Model
from transformers import ViTForImageClassification, ViTImageProcessor, pipeline
from gtts import gTTS
import gradio as gr
from PIL import Image
import numpy as np
import cv2
import os

# Define the model directory where config.json and model.safetensors are stored
model_directory = '/content/deepfake_model'

# Load the model from the safetensors file and the config.json
model = ViTForImageClassification.from_pretrained(
    model_directory,
    use_safetensors=True  # Use safetensors format for loading the model
)

# Manually create a ViTImageProcessor for the model
processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224-in21k")

# Create a prediction pipeline using the loaded model and processor
pipe = pipeline("image-classification", model=model, feature_extractor=processor)

def voice_message(prediction_label):
    if prediction_label == 'Fake':
        message = "Warning, this is a deepfake."
    else:
        message = "This image is real."

    # Generate speech using gTTS
    tts = gTTS(message)
    audio_file = "voice_message.mp3"
    tts.save(audio_file)
    return audio_file

def is_face_present(img: Image.Image) -> bool:
    # Convert PIL image to a format compatible with OpenCV
    cv_img = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    # Load pre-trained face detector from OpenCV
    face_cascade = cv2.CascadeClassifier(
        cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
    )
    gray = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.3, minNeighbors=5)
    return len(faces) > 0

def predict_image(img: Image.Image):
    try:
        # Validate that the image is properly loaded
        if img is None:
            raise ValueError("No image provided.")

        # Check if a face is present in the image
        if not is_face_present(img):
            return "Invalid input - No face detected", None

        # Get the top prediction from the pipeline
        prediction = pipe(img)[0]
        label = prediction['label']  # Expected to be 'Real' or 'Fake'
        score = prediction['score']

        # Generate a voice message MP3 based on the prediction
        audio_file = voice_message(label)

        result_text = f"Prediction: {label}, Score: {score:.4f}"
        return result_text, audio_file

    except Exception as e:
        # Capture any exceptions (e.g., file corruption, processing errors) and return an error message
        return f"Error: {str(e)}", None


# Gradio Interface for User to Upload an Image

In [ ]:
# Create Gradio interface for image upload and prediction with two outputs: text and audio
interface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil"),
    outputs=[ "text", "audio" ],
    title="Deepfake Detector",
    description="Upload an image to predict if it is real or a deepfake. A voice message will be generated based on the prediction."
)

# Launch the Gradio interface
interface.launch(debug=True)